# Politics — rolling wallet-selection (POC)

Walks the O2/Politics flow forward: instead of a **static** `COPY_DEFAULT`
wallet set, re-select the copy universe **every day** from the **USER
(point-in-time)** engine.

The wallet mask itself is recomputed **every day** from contract statistics
for markets resolved `≤ t − 14d` (the 14d buffer avoids resolution-time
ambiguity), applying `base_mask2` + `copyable_mask` — `buy_roi ≥ 0.05`,
`num_markets ≥ 30`, `max_drawdown/pnl ≤ 0.2`, market-PnL HHI `< 0.2`,
`total_notional ≥ $5k`, `median_dt ≤ t − 30d`, plus `buy_copyable_pnl > $1k`,
`buy_copyable_roi ≥ 0.1`, `copyable_pnl_factor ≥ 0.1`. Wallets that only
recently became active enter the pool as soon as they qualify; the
train/test split is used purely to tune scale, it never restricts the
selection data.

**Pre-selection + shared metrics.** The candidate universe is computed
**once** (the strictly monotone `num_markets`/`notional` criteria over the
full window plus the running-max envelope of `buy_copyable_pnl` — a complete
superset of any-day passers), so the daily metrics pass never touches
non-candidates. Each day the wallet metrics are then recomputed **once** and
shared by **all variants**; only the variant hygiene filters and alpha clip
differ:

- `UNL` — base mask only.
- `UNL_B` — + top-market PnL share ≤ 50% and max-drawdown/pnl ≤ 30%.
- `UNL_B_hhi` — + market-PnL HHI cap `< 0.30`.
- `UNL_p98` / `UNL_p95` / `UNL_p90` — inverse-variance alpha clipped at the
  98th / 95th / 90th percentile (vs p99 for the others).

Overlays: copy everything (`none`) vs only `price < 0.1` BUYs. Scale is tuned
on **validation only**, then applied unchanged to **test** (no test leakage).
Baselines: static `COPY_DEFAULT` and all-wallets `price < 0.1`.

In [1]:
# Setup: imports, paths, split configuration
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "polymarket" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PROJECT = ROOT
NOTEBOOKS = PROJECT / "notebooks"
WALLET = NOTEBOOKS / "wallet_selection"
SIGNAL_LAB = WALLET / "signal_lab"
HERE = SIGNAL_LAB / "onchain" / "politics"
CACHE_DIR = HERE / "rc_cache"

for p in (str(PROJECT), str(NOTEBOOKS), str(WALLET)):
    if p not in sys.path:
        sys.path.insert(0, p)

from signal_lab.filters import COPY_DEFAULT
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    select_scale,
    sizing_sharpe,
)
from signal_lab.stage1 import (
    _attach_copy_wallet_metrics,
    candidate_splits_for,
    load_stage1_data,
    split_data_at_dates,
)

SPLIT_KWARGS = {
    "train_end": "2026-02-01",
    "val_end": "2026-05-31",
    "test_start": "2026-06-01",
}
BUDGET = 10_000.0
SCALE_GRID = np.arange(0.1, 3.01, 0.1)

MAX_SHARDS = None  # full 16-shard run
TAG = "full" if MAX_SHARDS is None else f"shards{MAX_SHARDS}"

print(f"PROJECT: {PROJECT}")
print(f"HERE:    {HERE}")
print(f"PYTHON:  {sys.executable}")
print(f"TAG:     {TAG}  (MAX_SHARDS={MAX_SHARDS})")


PROJECT: /Users/vobornij/projects/polymarket
HERE:    /Users/vobornij/projects/polymarket/notebooks/wallet_selection/signal_lab/onchain/politics
PYTHON:  /Users/vobornij/projects/polymarket/.venv/bin/python
TAG:     full  (MAX_SHARDS=None)


## Method

**Universe (pre-selection, once).** The candidate set is the complete superset
of any-day mask passers, built once from the full window: `num_markets ≥ 30`
and `total_notional ≥ $5k` are monotone, and `buy_copyable_pnl` is checked on
its running-max envelope (`> $1k`). This keeps the daily metrics pass to
~1.7k wallets instead of ~34k while never dropping a wallet that would pass on
any day. The panel axis runs from the first trade day to the last evaluation
day; a trade's PnL is attributed to the day `max(end_date_iso, market_close)`
of its market.

**Rolling selection.** On each evaluation day `t`, the wallet metrics are
recomputed **once** from all contracts resolved `≤ t − 14d` and shared across
every variant; `base_mask2` + `copyable_mask` (incl. the now-effective
`median_dt ≤ t − 30d`) is applied, and the selected wallets' BUYs on day `t`
are copied. All statistics use strictly pre-`t` information — causal by
construction, no finite lookback.

**Scaling.** Each selected wallet gets `alpha = 1` (equal) or an
inverse-variance weight from its daily PnL over the same history (shrunken
toward the cross-wallet mean, p99-clipped by default — p98/p95/p90 variants
clip lower, mean-1 normalised, capped at 4.0). The backtest fires
`scale × alpha × copyable_qty` through the capital-constrained
`capital_constrained_sim` ($10k budget); `scale` is grid-searched on
**validation** by daily Sharpe and applied unchanged to **test**.

In [ ]:
# Engine: universe builder + causal rolling-selection panel
from polymarket_analysis.wallet_selection.volatility import _wallet_metrics_from_buckets


def build_universe(df_full, split_kwargs):
    wallets_all = set(df_full["wallet"].unique())
    splits = candidate_splits_for(df_full, wallets_all, **split_kwargs)
    buys = pd.concat(splits.values(), ignore_index=True)
    buys["split"] = np.repeat(
        ["train", "val", "test"], [len(splits[k]) for k in ("train", "val", "test")]
    )
    buys["day"] = pd.to_datetime(buys["dt"], utc=True).dt.floor("1D")
    rel = np.maximum(
        pd.to_datetime(buys["end_date_iso"], utc=True).values.astype("datetime64[ns]"),
        pd.to_datetime(buys["market_close"], utc=True).values.astype("datetime64[ns]"),
    )
    buys["rel_day"] = pd.DatetimeIndex(rel).tz_localize("UTC").floor("1D")
    buys["score1"] = 1.0
    buys["alpha_equal"] = 1.0
    return buys



# ── Point-in-time USER mask machinery ────────────────────────────────────────
# The USER variants re-select the wallet set EVERY day: wallet metrics are
# recomputed from all contracts resolved <= t - 14d, then ``base_mask2`` +
# ``copyable_mask`` is applied.  This is causal by construction and lets
# wallets that only recently became active enter as soon as they qualify.
#
# Pre-selection + shared metrics: ``user_candidate_wallets`` (monotone criteria
# over the full window) is a complete superset of any-day passers and bounds
# the daily pass to ~1.7k wallets; per decision day the mask metrics are
# computed ONCE and shared by every variant (only hygiene filters and the
# alpha clip differ between variants).
DAY0 = pd.Timestamp("2025-01-01", tz="UTC")


def build_buckets(df_full):
    """One-time bucket aggregation (identical to compute_wallet_metrics) with
    the market resolution day (``last_condition_trade_ts``) attached.  Row
    order is preserved so per-wallet drawdown stays chronological."""
    res_day = (
        df_full.groupby("condition_id")["last_condition_trade_ts"].max().dt.floor("1D")
    )
    tmp = df_full.copy()
    tmp["dt_floored"] = tmp["dt"].dt.floor("5min")
    buckets = (
        tmp.groupby(
            ["wallet", "dt_floored", "condition_id", "side"], sort=False, observed=True
        )
        .agg(
            notional=("notional", "sum"),
            pnl=("pnl", "sum"),
            copyable_pnl=("copyable_pnl", "sum"),
            quantity=("quantity", "sum"),
            copyable_qty_sum=("copyable_qty", "sum"),
            trade_count=("pnl", "size"),
            avail_copy_total_vol=("avail_copy_total_vol", "max"),
        )
        .reset_index()
    )
    buckets["copyable_qty"] = np.minimum(
        buckets["copyable_qty_sum"], buckets["avail_copy_total_vol"]
    )
    m = buckets["copyable_qty_sum"] > 0
    buckets.loc[m, "copyable_pnl"] *= (
        buckets.loc[m, "copyable_qty"] / buckets.loc[m, "copyable_qty_sum"]
    )
    buckets = buckets.drop(columns="copyable_qty_sum")
    buckets = buckets[buckets["notional"] > 0].copy()
    buckets["res_i"] = (
        (buckets["condition_id"].map(res_day) - DAY0).dt.days.astype(int)
    )
    return buckets


def _user_mask_metrics_from_buckets(buckets):
    """Per-wallet metrics used by the USER mask, computed from a bucket frame.

    This is a stripped-down ``_wallet_metrics_from_buckets``: only the columns
    the mask reads, all vectorized (groupby.cumsum/cummax for drawdown, no
    per-wallet Python loops, no top-N / volatility passes).  Verified to match
    ``_wallet_metrics_from_buckets`` exactly on the USER-mask inputs.
    """
    df = buckets.copy()
    g = df.groupby("wallet", sort=False)
    res = pd.DataFrame(
        {"wallet": df["wallet"].drop_duplicates().to_numpy()}
    ).set_index("wallet")
    res["num_markets"] = g["condition_id"].nunique()
    res["total_notional"] = g["notional"].sum()
    res["total_pnl"] = g["pnl"].sum()
    res["copyable_pnl"] = g["copyable_pnl"].sum()
    res["median_dt"] = g["dt_floored"].median()

    df["_cum"] = g["pnl"].cumsum()
    peak = np.maximum(
        df.groupby("wallet", sort=False)["_cum"].cummax(), 0.0
    )
    df["_dd"] = peak - df["_cum"]
    res["max_drawdown"] = df.groupby("wallet", sort=False)["_dd"].max()
    with np.errstate(divide="ignore", invalid="ignore"):
        res["max_drawdown_to_pnl"] = np.where(
            res["total_pnl"] > 0, res["max_drawdown"] / res["total_pnl"], np.nan
        )

    piv = df.pivot_table(
        index="wallet", columns="side",
        values=["pnl", "quantity", "notional", "copyable_qty", "copyable_pnl"],
        aggfunc="sum", sort=False, fill_value=0.0,
    )

    def side_sum(var, side):
        return piv[(var, side)] if (var, side) in piv.columns else 0.0

    res["buy_pnl"] = side_sum("pnl", "BUY")
    res["buy_quantity"] = side_sum("quantity", "BUY")
    res["buy_notional"] = side_sum("notional", "BUY")
    res["buy_copyable_pnl"] = side_sum("copyable_pnl", "BUY")
    res["buy_copyable_quantity"] = side_sum("copyable_qty", "BUY")
    res["buy_roi"] = np.where(
        res["buy_notional"] > 0, res["buy_pnl"] / res["buy_notional"], 0.0
    )
    res["buy_copyable_notional"] = np.where(
        res["buy_quantity"] > 0,
        res["buy_notional"] * (res["buy_copyable_quantity"] / res["buy_quantity"]),
        np.nan,
    )

    mp = df.groupby(["wallet", "condition_id"], sort=False)["pnl"].sum().abs()
    mtotal = mp.groupby("wallet", sort=False).sum()
    nz = mtotal > 0
    hhi = (mp / mtotal).pow(2).groupby("wallet", sort=False).sum()
    res["market_pnl_hhi"] = np.where(nz, hhi, np.nan)
    return res.reset_index()


def user_mask_bools(wm, med_cut):
    """``base_mask2`` + ``copyable_mask`` on a point-in-time metrics frame.

    ``median_dt <= med_cut`` is now EFFECTIVE because the metrics window is
    recomputed per decision day (contracts resolved <= t - 14d), so a wallet
    whose median trade is younger than ``med_age_days`` is excluded.
    """
    w = wm.copy()
    w["buy_copyable_roi"] = (
        w["buy_copyable_pnl"] / w["buy_copyable_notional"].replace(0, np.nan)
    )
    w["copyable_pnl_factor"] = np.clip(
        w["copyable_pnl"] / w["total_pnl"].replace(0, np.nan), 0, 1.0
    ).fillna(0.0)
    base_mask2 = (
        (w["buy_roi"] >= 0.05)
        & (w["num_markets"] >= 30)
        & (w["max_drawdown_to_pnl"] <= 0.2)
        & (w["market_pnl_hhi"].fillna(0.20) < 0.2)
        & (w["total_notional"] >= 5000)
        & (w["median_dt"] <= med_cut)
    )
    copyable_mask = (
        (w["buy_copyable_pnl"] > 1000)
        & (w["buy_copyable_roi"] >= 0.1)
        & (w["copyable_pnl_factor"] >= 0.1)
    )
    return base_mask2 & copyable_mask


def user_candidate_wallets(
    buckets, *, min_markets=30, min_notional=5000.0, min_buy_copyable_pnl=1000.0
):
    """Complete superset of any-day mask passers: the strictly monotone
    criteria (``num_markets``, ``total_notional``) are checked at the full
    window; the non-monotone ``buy_copyable_pnl`` is checked on its running-max
    envelope.  No wallet that passes on any day is missed."""
    buyb = buckets[buckets["side"] == "BUY"]
    full = buckets.groupby("wallet").agg(
        n_mkt=("condition_id", "nunique"), tot_notional=("notional", "sum")
    )
    env = (
        buyb.groupby(["wallet", "res_i"], sort=True)["copyable_pnl"]
        .sum()
        .groupby("wallet")
        .cumsum()
        .groupby("wallet")
        .cummax()
    )
    sel = (
        (full["n_mkt"] >= min_markets)
        & (full["tot_notional"] >= min_notional)
        & (env.groupby("wallet").max() > min_buy_copyable_pnl)
    )
    return set(full.index[sel])


def user_rolling_copy_panels(buys, buckets, variants,
                             *, res_buffer_days=14, med_age_days=30,
                             alpha_max=4.0, shrink_lambda=30.0):
    """Point-in-time USER wallet selection for a family of variants.

    ``variants`` maps a panel name to a criteria dict; each dict may set the
    hygiene filters ``max_top_market_share`` / ``max_dd_ratio`` /
    ``max_market_pnl_hhi`` and the alpha clip ``alpha_clip_percentile``.

    On each decision day ``t`` the wallet metrics are recomputed ONCE from all
    contracts resolved ``<= t - res_buffer_days`` and shared across every
    variant; ``base_mask2`` + ``copyable_mask`` (incl. ``median_dt <=
    t - med_age_days``) is applied once, then each variant applies its hygiene
    filters and inverse-variance alpha.  The candidate universe is a complete
    superset of any-day passers.  Returns ``{panel: (selr, sel_day_count)}``.
    """
    t0 = time.time()
    vnames = list(variants)
    need_dd = any(variants[v].get("max_dd_ratio") is not None for v in vnames)
    need_share = any(variants[v].get("max_top_market_share") is not None for v in vnames)
    need_hhi = any(variants[v].get("max_market_pnl_hhi") is not None for v in vnames)

    eval_days = pd.Index(sorted(set(buys.loc[buys["split"] != "train", "day"].unique())))
    n_axis = (eval_days.max() - DAY0).days + 1
    days = pd.date_range(DAY0, periods=n_axis, freq="1D", tz="UTC")
    day_to_i = {d: i for i, d in enumerate(days)}

    cand = user_candidate_wallets(buckets)
    print(f"  [USER candidates] complete superset: {len(cand)} wallets")
    b_cand = buckets[buckets["wallet"].isin(cand)].copy()
    w_to_i = {w: i for i, w in enumerate(sorted(cand))}
    n_w = len(w_to_i)
    res_i_cand = b_cand["res_i"].to_numpy()

    rr = buys[buys["wallet"].isin(w_to_i)].copy()
    rr["w"] = rr["wallet"].map(w_to_i)
    rr["d_rel"] = rr["rel_day"].map(day_to_i)
    rr = rr[rr["d_rel"].notna() & (rr["d_rel"] >= 0) & (rr["d_rel"] < n_axis)]
    pnl_day = np.zeros((n_axis, n_w))
    pg = rr.groupby(["d_rel", "w"])["copyable_pnl"].sum().reset_index()
    pnl_day[pg["d_rel"].astype(int).to_numpy(), pg["w"].astype(int).to_numpy()] = (
        pg["copyable_pnl"].to_numpy()
    )
    cpnl = pnl_day.cumsum(axis=0)
    cpnl2 = (pnl_day**2).cumsum(axis=0)

    m_pnl = rr.groupby(
        ["wallet", "condition_id", "d_rel"], sort=False
    )["copyable_pnl"].sum().reset_index()
    m_arr_day = m_pnl["d_rel"].astype(int).to_numpy()
    m_arr_w = m_pnl["wallet"].map(w_to_i).astype(int).to_numpy()
    m_arr_pnl = m_pnl["copyable_pnl"].to_numpy()
    m_arr_cid = m_pnl["condition_id"].to_numpy()

    n_t = len(buys)
    sel_v = {v: np.zeros(n_t, dtype=bool) for v in vnames}
    eq_v = {v: np.zeros(n_t) for v in vnames}
    iv_v = {v: np.zeros(n_t) for v in vnames}
    cnt_v = {v: pd.Series(0, index=eval_days, dtype=int) for v in vnames}

    w_idx_f = buys["wallet"].map(w_to_i).astype(float).to_numpy()
    w_known = np.isfinite(w_idx_f)
    w_idx_all = np.where(w_known, w_idx_f, 0).astype(np.int64)
    d_idx_all = (buys["day"] - DAY0).dt.days.to_numpy()

    for t in eval_days:
        ti = day_to_i[t]
        hi = ti - 1
        elig = b_cand[res_i_cand <= ti - res_buffer_days]
        if elig.empty:
            continue
        wm_t = _user_mask_metrics_from_buckets(elig)
        ok = user_mask_bools(wm_t, t - pd.Timedelta(days=med_age_days)).to_numpy()
        if not ok.any():
            continue
        w_sel0 = wm_t.loc[ok, "wallet"].to_numpy()
        wi0 = wm_t.loc[ok, "wallet"].map(w_to_i).to_numpy()
        pnl_sum0 = cpnl[hi][wi0]

        # shared hygiene inputs (computed once for all variants that need them)
        dd_ratio = None
        top_share = None
        hhi_val = None
        if need_dd:
            win = pnl_day[: hi + 1][:, wi0]
            cum = win.cumsum(axis=0)
            runmax = np.maximum.accumulate(cum, axis=0)
            maxdd = (runmax - cum).max(axis=0)
            with np.errstate(divide="ignore", invalid="ignore"):
                dd_ratio = np.where(pnl_sum0 > 0, maxdd / pnl_sum0, np.inf)
        if need_share:
            msel = (m_arr_day > -1) & (m_arr_day <= hi)
            sub_w, sub_p = m_arr_w[msel], m_arr_pnl[msel]
            tot = np.zeros(n_w)
            np.add.at(tot, sub_w, sub_p)
            top = np.full(n_w, -np.inf)
            np.maximum.at(top, sub_w, sub_p)
            with np.errstate(divide="ignore", invalid="ignore"):
                share = np.where(tot > 0, top / tot, np.inf)
            top_share = share[wi0]
        if need_hhi:
            msel = (m_arr_day > -1) & (m_arr_day <= hi)
            wsub = pd.DataFrame(
                {"w": m_arr_w[msel], "c": m_arr_cid[msel], "a": np.abs(m_arr_pnl[msel])}
            )
            per = wsub.groupby(["w", "c"], sort=False)["a"].sum()
            per_w = per.groupby("w", sort=False).sum()
            sq = (per / per_w).pow(2).groupby("w", sort=False).sum()
            hhi = pd.Series(0.20, index=np.arange(n_w))
            hhi.loc[sq.index] = sq.to_numpy()
            hhi_val = hhi.to_numpy()[wi0]

        rows_t = d_idx_all == ti
        wnum = w_idx_all[rows_t]
        known = w_known[rows_t]

        for v in vnames:
            kw = variants[v]
            keep = np.ones(len(wi0), dtype=bool)
            if kw.get("max_dd_ratio") is not None:
                keep &= dd_ratio <= kw["max_dd_ratio"]
            if kw.get("max_top_market_share") is not None:
                keep &= top_share <= kw["max_top_market_share"]
            if kw.get("max_market_pnl_hhi") is not None:
                keep &= hhi_val < kw["max_market_pnl_hhi"]
            wi = wi0[keep]
            pnl_sum = pnl_sum0[keep]
            if wi.size == 0:
                continue
            cnt_v[v][t] = wi.size

            # inverse-variance alpha over resolved-pnl history up to t-1
            W = hi + 1
            pnl2_sum = cpnl2[hi][wi]
            mean = pnl_sum / W
            var = np.clip(pnl2_sum / W - mean**2, 0.0, None)
            var_cs = float(np.mean(var)) if var.size else 1.0
            var_cs = var_cs if var_cs > 0 else 1.0
            var_shrunk = (W * var + shrink_lambda * var_cs) / (W + shrink_lambda)
            raw = 1.0 / np.sqrt(np.maximum(var_shrunk, 1e-12))
            clip_p = kw.get("alpha_clip_percentile", 99.0)
            raw = np.clip(raw, 0.0, np.percentile(raw, clip_p))
            m = raw.mean()
            a_iv = (raw / m if m > 0 else np.ones(len(wi))).clip(0.0, alpha_max)

            a_full = np.zeros(n_w)
            a_full[wi] = a_iv
            sel_ok = known & np.isin(wnum, wi)
            sel_v[v][rows_t] = sel_ok
            eq_v[v][rows_t] = np.where(sel_ok, 1.0, 0.0)
            iv_v[v][rows_t] = np.where(known, a_full[wnum], 0.0)

    out = {}
    for v in vnames:
        mask = sel_v[v] & (buys["split"].to_numpy() != "train")
        idx = np.nonzero(mask)[0]
        selr = buys.iloc[idx].copy()
        selr["alpha_equal"] = eq_v[v][idx]
        selr["alpha_invvol"] = iv_v[v][idx]
        selr["sel_day"] = selr["day"].map(cnt_v[v]).fillna(0).astype(int)
        nz = cnt_v[v][cnt_v[v] > 0]
        print(
            f"[USER rolling] {v}: top_share={variants[v].get('max_top_market_share')} "
            f"dd={variants[v].get('max_dd_ratio')} hhi={variants[v].get('max_market_pnl_hhi')} "
            f"clip={variants[v].get('alpha_clip_percentile', 99.0)} "
            f"wallets_axis={n_w} mean_sel/day={nz.mean():.0f} rows={len(selr):,}"
        )
        out[v] = (
            selr[["wallet", "condition_id", "dt", "price", "copyable_qty",
                  "copyable_pnl", "end_date_iso", "market_close", "split",
                  "day", "sel_day", "score1", "alpha_equal", "alpha_invvol"]],
            cnt_v[v],
        )
    print(f"  [USER panels] {time.time()-t0:.1f}s for {len(vnames)} variants")
    return out


In [34]:
# Load trade data once, build the all-wallets universe
t0 = time.time()
df_full, _dt, _dv, _dtest, wm, hm = load_stage1_data(
    tags={"Politics"}, max_shards=MAX_SHARDS, **SPLIT_KWARGS,
)

# initial filter
eligible_wallets = df_full[(df_full['side'] == 'BUY')
        & (df_full['copyable_pnl'] > 0)].groupby('wallet').agg(
            copyable_pnl=('copyable_pnl', 'sum'),
            market_count=('condition_id', 'nunique'),
            trade_count=('wallet', 'count'),
        ).query('copyable_pnl > 1000 and market_count >= 30')

print(len(eligible_wallets), eligible_wallets['trade_count'].sum())

df_full = df_full[df_full['wallet'].isin(eligible_wallets.index)].copy()

print(len(df_full))

buys = build_universe(df_full, SPLIT_KWARGS)
print(
    f"[load] {time.time()-t0:.1f}s buys={len(buys):,} "
    f"wallets_all={len(set(df_full['wallet'])):,}"
)


Markets: 2165960
Filtered markets for {'Politics'}: 44273
Loading 16 trade shards...
Total trades loaded: 13,282,697
Unique wallets: 34,344
Date range: 2025-01-01 00:00:59+00:00 -> 2026-08-03 16:01:48+00:00
split_data_at_dates: train_end=2026-02-01 val_end=2026-05-31 test_start=2026-06-01
  Train:  4,960,227 trades  (7,817 markets)
  Val:    5,345,970 trades  (7,287 markets)
  Test:   2,976,500 trades  (5,947 markets)
  Total: 13,282,697 trades  (21,051 markets)


/Users/vobornij/projects/polymarket/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

invalid value encountered in sqrt



2020 2257118
10656102
split_data_at_dates: train_end=2026-02-01 val_end=2026-05-31 test_start=2026-06-01
  Train:  2,275,464 trades  (7,800 markets)
  Val:    2,557,170 trades  (7,251 markets)
  Test:   1,555,810 trades  (5,792 markets)
  Total:  6,388,444 trades  (20,843 markets)
[load] 91.0s buys=6,388,444 wallets_all=2,020


In [35]:
# Build the USER selection panels (point-in-time wallet mask; all variants
# share one daily metrics pass), cached by data tag.
BUCKETS = build_buckets(df_full)
print(f"[buckets] {len(BUCKETS):,} bucket-rows (point-in-time USER engine)")

USER_RT_CRIT = dict(res_buffer_days=14, med_age_days=30)
USER_RT_VARIANTS = {
    "UNL": dict(),
    "UNL_B": dict(max_top_market_share=0.5, max_dd_ratio=0.3),
    "UNL_B_hhi": dict(max_top_market_share=0.5, max_dd_ratio=0.3,
                      max_market_pnl_hhi=0.30),
    "UNL_p98": dict(alpha_clip_percentile=98.0),
    "UNL_p95": dict(alpha_clip_percentile=95.0),
    "UNL_p90": dict(alpha_clip_percentile=90.0),
}
PANEL_KEYS = list(USER_RT_VARIANTS)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_paths = {k: CACHE_DIR / f"rt_{TAG}_{k}.parquet" for k in PANEL_KEYS}
sd_paths = {k: CACHE_DIR / f"rt_{TAG}_{k}_selday.parquet" for k in PANEL_KEYS}


def _load_or_build():
    if all(p.exists() for p in cache_paths.values()):
        panels, sel_day = {}, {}
        for k in PANEL_KEYS:
            panels[k] = pd.read_parquet(cache_paths[k])
            sd = pd.read_parquet(sd_paths[k])["n"]
            sd.index = pd.to_datetime(sd.index, utc=True)
            sel_day[k] = sd
            print(f"[cache] {k} loaded ({len(panels[k]):,} rows)")
        return panels, sel_day
    built = user_rolling_copy_panels(buys, BUCKETS, USER_RT_VARIANTS)
    panels, sel_day = {}, {}
    for k in PANEL_KEYS:
        panels[k], sel_day[k] = built[k]
        panels[k].to_parquet(cache_paths[k])
        sel_day[k].to_frame("n").to_parquet(sd_paths[k])
        print(f"[cache] {k} saved ({len(panels[k]):,} rows)")
    return panels, sel_day


t1 = time.time()
panels, sel_day = _load_or_build()
print(f"[panels] {time.time()-t1:.1f}s  keys={sorted(panels)}")


[buckets] 7,189,610 bucket-rows (point-in-time USER engine)
[cache] UNL loaded (49,705 rows)
[cache] UNL_B loaded (25,776 rows)
[cache] UNL_B_hhi loaded (25,705 rows)
[cache] UNL_p98 loaded (49,705 rows)
[cache] UNL_p95 loaded (49,705 rows)
[cache] UNL_p90 loaded (49,705 rows)
[panels] 0.1s  keys=['UNL', 'UNL_B', 'UNL_B_hhi', 'UNL_p90', 'UNL_p95', 'UNL_p98']


In [38]:
# Parallel grid: panel x alpha scheme x overlay; scale tuned on val, applied to test
from joblib import Parallel, delayed

SIM_COLS = ["dt", "price", "copyable_qty", "copyable_pnl",
            "end_date_iso", "market_close", "score1"]


def run_sim(frame, scale, alpha_col, cost_bps=0.0):
    res = capital_constrained_sim(frame, "score1", BUDGET, scale, cost_bps,
                                  alpha_col=alpha_col)
    daily = res["daily_pnl"]
    cum = daily.cumsum()
    return {
        "scale": scale,
        "trades": int(res["trades"]),
        "net_pnl": round(float(res["net_pnl"]), 2),
        "peak_used": round(float(res["peak_used"]), 2),
        "mean_used": round(float(res["mean_used"]), 2),
        "sharpe_daily": round(sizing_sharpe(daily, 365.0), 3),
        "max_dd": round(float((cum.cummax() - cum).max()) if len(cum) else 0.0, 2),
        "daily": daily,
    }


def _task(args):
    name, fr_v, fr_t, alpha_col = args
    if fr_v.empty or fr_t.empty:
        return None
    best_scale, _ = select_scale(
        fr_v, "score1", BUDGET, SCALE_GRID, 0.0, primary="sharpe_daily",
    )
    s = float(best_scale)
    rv = run_sim(fr_v, s, alpha_col)
    rt = run_sim(fr_t, s, alpha_col)
    return {
        "config": name, "alpha_col": alpha_col, "val_scale": s,
        "val_sharpe": rv["sharpe_daily"], "val_pnl": rv["net_pnl"],
        "val_trades": rv["trades"], "val_mean_used": rv["mean_used"],
        "val_max_dd": rv["max_dd"],
        "test_sharpe": rt["sharpe_daily"], "test_pnl": rt["net_pnl"],
        "test_trades": rt["trades"], "test_mean_used": rt["mean_used"],
        "test_max_dd": rt["max_dd"],
        "test_daily": rt["daily"],
    }


tasks = []
for key, selr in panels.items():
    for alpha_scheme in ("equal", "invvol"):
        acol = f"alpha_{alpha_scheme}"
        for overlay in ("none", "p<0.1"):
            fr_v = selr[selr["split"] == "val"]
            fr_t = selr[selr["split"] == "test"]
            if overlay == "p<0.1":
                fr_v = fr_v[fr_v["price"] < 0.1]
                fr_t = fr_t[fr_t["price"] < 0.1]
            fr_v = fr_v[SIM_COLS + [acol]]
            fr_t = fr_t[SIM_COLS + [acol]]
            tasks.append((f"{key}_{alpha_scheme}_{overlay}", fr_v, fr_t, acol))
print(f"[tasks] {len(tasks)} configs")

t1 = time.time()
grid_rows = [
    r for r in Parallel(n_jobs=8, prefer="processes")(delayed(_task)(t) for t in tasks)
    if r is not None
]
grid = pd.DataFrame(grid_rows).sort_values("val_sharpe", ascending=False)
print(f"[grid sims parallel] {time.time()-t1:.1f}s")
view = ["config", "val_scale", "val_sharpe", "val_pnl",
        "test_sharpe", "test_pnl", "test_trades", "test_max_dd"]
print(grid[view].round(3).to_string(index=False))


[tasks] 24 configs
[grid sims parallel] 2.5s
                config  val_scale  val_sharpe  val_pnl  test_sharpe  test_pnl  test_trades  test_max_dd
    UNL_B_invvol_p<0.1        0.3       2.001  3045.42       -1.331   -439.48         1505       618.35
UNL_B_hhi_invvol_p<0.1        0.3       2.001  3045.42       -1.330   -438.91         1503       617.78
 UNL_B_hhi_equal_p<0.1        0.3       1.778 10024.10        0.075    122.61         1503      1806.94
     UNL_B_equal_p<0.1        0.3       1.778 10024.10        0.074    121.22         1505      1808.34
  UNL_p90_invvol_p<0.1        0.1       1.595  2392.26        0.373    400.33         2414       564.82
  UNL_p98_invvol_p<0.1        0.1       1.595  2391.85        0.373    400.28         2414       564.74
  UNL_p95_invvol_p<0.1        0.1       1.595  2392.04        0.373    400.30         2414       564.78
      UNL_invvol_p<0.1        0.1       1.595  2391.75        0.373    400.28         2414       564.74
  UNL_B_hhi_equal_n

In [39]:
# Baselines: static COPY_DEFAULT set and all-wallets price<0.1, same protocol
base_slim = SIM_COLS + ["alpha_equal"]
static_w = set(COPY_DEFAULT(wm, hm))
print(f"COPY_DEFAULT wallets: {len(static_w)}")

base_tasks = []
for name, mask in (
    ("static_copy_default", buys["wallet"].isin(static_w)),
    ("price_lt_0p1_all", buys["price"] < 0.1),
):
    fr_v = buys[mask & (buys["split"] == "val")][base_slim].copy()
    fr_t = buys[mask & (buys["split"] == "test")][base_slim].copy()
    base_tasks.append((name, fr_v, fr_t, "alpha_equal"))

t1 = time.time()
base_rows = [
    r for r in Parallel(n_jobs=4, prefer="processes")(delayed(_task)(t) for t in base_tasks)
    if r is not None
]
base_df = pd.DataFrame(base_rows)
print(f"[baselines parallel] {time.time()-t1:.1f}s")
print(base_df[view].round(3).to_string(index=False))


COPY_DEFAULT wallets: 111
[baselines parallel] 10.3s
             config  val_scale  val_sharpe   val_pnl  test_sharpe  test_pnl  test_trades  test_max_dd
static_copy_default        0.6       0.415   9532.42       -0.186  -1021.74         4730      5966.11
   price_lt_0p1_all        0.9       0.502 118015.48       -0.648 -11592.02        60159     11698.08


## Results

The grid table above is sorted by **validation** Sharpe (the honest selector).
`test_sharpe` / `test_pnl` use the validation-tuned scale, so a configuration
that ranks well on val but poorly on test is exactly the failure mode the
POC is designed to expose. Baselines follow the same protocol.


In [40]:
# Block-bootstrap Sharpe CIs on test daily PnL (block=7d, n=1000, seed 42)
daily_test = {r["config"]: r["test_daily"] for _, r in grid.iterrows()}
daily_base = {r["config"]: r["test_daily"] for _, r in base_df.iterrows()}
both = {**daily_test, **daily_base}

targets = ["UNL_B_hhi_equal_p<0.1", "UNL_B_equal_p<0.1", "UNL_B_invvol_p<0.1",
           "UNL_B_hhi_equal_none", "UNL_equal_p<0.1", "UNL_p98_equal_p<0.1",
           "UNL_p95_equal_p<0.1", "UNL_p90_equal_p<0.1",
           "static_copy_default", "price_lt_0p1_all"]
boot_rows = []
for cfg in targets:
    d = both[cfg]
    point, lo, hi = block_bootstrap_sharpe(d, block_size=7, n_iter=1000, seed=42)
    boot_rows.append({
        "config": cfg,
        "sharpe": round(point, 3),
        "ci_lo": round(lo, 3),
        "ci_hi": round(hi, 3),
        "test_pnl": round(float(d.sum()), 2),
    })
print(pd.DataFrame(boot_rows).round(3).to_string(index=False))


               config  sharpe  ci_lo  ci_hi  test_pnl
UNL_B_hhi_equal_p<0.1   0.075 -4.329  0.562    122.61
    UNL_B_equal_p<0.1   0.074 -4.330  0.562    121.22
   UNL_B_invvol_p<0.1  -1.331 -4.062  0.979   -439.48
 UNL_B_hhi_equal_none   1.424 -2.876  1.312   1829.96
      UNL_equal_p<0.1   0.536 -4.528  1.612   1011.82
  UNL_p98_equal_p<0.1   0.536 -4.528  1.612   1011.82
  UNL_p95_equal_p<0.1   0.536 -4.528  1.612   1011.82
  UNL_p90_equal_p<0.1   0.536 -4.528  1.612   1011.82
  static_copy_default  -0.186 -3.839  1.261  -1021.74
     price_lt_0p1_all  -0.648 -3.686  1.080 -11592.02


In [41]:
# Plots: cumulative test PnL and daily selected-wallet count
import plotly.graph_objects as go

b_family = ["UNL_equal_p<0.1", "UNL_B_equal_p<0.1", "UNL_B_hhi_equal_p<0.1",
            "UNL_p98_equal_p<0.1", "UNL_p95_equal_p<0.1", "UNL_p90_equal_p<0.1"]

fig = go.Figure()
for cfg in b_family + ["static_copy_default", "price_lt_0p1_all"]:
    d = both[cfg].sort_index()
    fig.add_trace(go.Scatter(x=d.index, y=d.cumsum(), name=cfg, mode="lines"))
fig.update_layout(
    title=f"Test cumulative PnL ($10k budget, val-tuned scale) — Politics",
    template="plotly_dark",
    xaxis_title="resolution day",
    yaxis_title="cumulative PnL ($)",
)
fig.show()

fig2 = go.Figure()
for key in ("UNL", "UNL_B", "UNL_B_hhi", "UNL_p98"):
    sd = sel_day[key]
    fig2.add_trace(go.Scatter(x=sd.index, y=sd.values, name=key, mode="lines"))
fig2.update_layout(
    title="Selected wallets per day (criterion B)",
    template="plotly_dark",
    xaxis_title="day",
    yaxis_title="wallets selected",
)
fig2.show()


In [42]:
# Write results for the record
grid_out = grid.drop(columns=["test_daily"])
base_out = base_df.drop(columns=["test_daily"])
grid_out.to_csv(HERE / "rc_politics_grid.csv", index=False)
base_out.to_csv(HERE / "rc_politics_baselines.csv", index=False)
best = grid.iloc[0]
(HERE / "rc_politics_summary.json").write_text(json.dumps({
    "max_shards": MAX_SHARDS,
    "best_config": best["config"],
    "best": {
        k: (round(v, 4) if isinstance(v, float) else v)
        for k, v in best.drop(labels=["test_daily"]).to_dict().items()
    },
    "grid": grid_out.to_dict(orient="records"),
    "baselines": base_out.to_dict(orient="records"),
}, indent=2, default=str))
print("wrote rc_politics_grid.csv / rc_politics_baselines.csv / rc_politics_summary.json")
print(f"best on val: {best['config']} -> test sharpe {best['test_sharpe']}, "
      f"test pnl {best['test_pnl']}")


wrote rc_politics_grid.csv / rc_politics_baselines.csv / rc_politics_summary.json
best on val: UNL_B_invvol_p<0.1 -> test sharpe -1.331, test pnl -439.48


## Headline summary

Selection is the daily point-in-time USER mask over the pre-selected candidate
superset (~1.7k wallets), variants `UNL`, `UNL_B`, `UNL_B_hhi`, `UNL_p98`,
`UNL_p95`, `UNL_p90`, each under `equal`/`invvol` × `none`/`price<0.1`. Scale
is tuned on validation (2026-02-01 → 2026-05-31) and applied unchanged to
test (2026-06-01 → 2026-08-03). *Results refreshed by the latest full run —
see the grid, bootstrap and baseline tables above.*

## Long-term evaluation (rolling, $10k, 2025-06 → present)

The test split above covers ≈2 months (2026-06-01 → 2026-08-03). This section
re-runs the same rolling copy universe over the **full available horizon**:
evaluation starts **2025-06-01** and rolls forward to the last data day
(**2026-08-03**) — a ≈14-month sample.

**Split.** Markets resolving on/before 2025-06-01 are train (only used as the
causal selection lookback); everything after is evaluation — there is no
separate validation period, so scale is chosen **walk-forward** instead.

**Walk-forward scale.** Every 14 days the grid (0.1–3.0 by 0.1, daily-Sharpe
objective) is re-tuned on the trailing 120 days of that config's *realized*
copy trades (strictly pre-trade-day: resolutions < decision day; tuned on a
40k-trade subsample). Each day's trades fire at that day's selected scale,
all inside **one continuous $10k `capital_constrained_sim`** so capital
carries across days. Before the first retune (and whenever the trailing
window is too thin) a default scale of 0.5 is used.

**Panels.** The same six USER point-in-time variants as the main split
(`UNL`, `UNL_B`, `UNL_B_hhi`, `UNL_p98`, `UNL_p95`, `UNL_p90`), recomputed
daily with the 14-day resolution buffer; panels are cached in
`rc_cache/rt_lt_*`. The `price<0.1` overlay is applied to all configs plus a
no-overlay contrast for the p-variants and `UNL_B_hhi`. Trades whose market
resolves after the data end are excluded (their copy PnL is not yet realized).

In [43]:
# Long-term panels: LT split, USER point-in-time selection (all variants), cached
SPLIT_KWARGS_LT = {"train_end": "2025-06-01", "val_end": None, "test_start": "2025-06-01"}
LT_START = pd.Timestamp("2025-06-01", tz="UTC")
DATA_END = pd.Timestamp("2026-08-03", tz="UTC")
LT_PANEL_KEYS = list(USER_RT_VARIANTS)

buys_lt = build_universe(df_full, SPLIT_KWARGS_LT)
print(f"buys_lt={len(buys_lt):,} wallets_all={len(set(df_full['wallet'])):,}")

cache_paths = {k: CACHE_DIR / f"rt_lt_{TAG}_{k}.parquet" for k in LT_PANEL_KEYS}
sd_paths = {k: CACHE_DIR / f"rt_lt_{TAG}_{k}_selday.parquet" for k in LT_PANEL_KEYS}


def _load_or_build_lt():
    if all(p.exists() for p in cache_paths.values()):
        panels, sel_day = {}, {}
        for k in LT_PANEL_KEYS:
            panels[k] = pd.read_parquet(cache_paths[k])
            sd = pd.read_parquet(sd_paths[k])["n"]
            sd.index = pd.to_datetime(sd.index, utc=True)
            sel_day[k] = sd
            print(f"[cache] {k} loaded ({len(panels[k]):,} rows)")
        return panels, sel_day
    built = user_rolling_copy_panels(buys_lt, BUCKETS, USER_RT_VARIANTS)
    panels, sel_day = {}, {}
    for k in LT_PANEL_KEYS:
        panels[k], sel_day[k] = built[k]
        panels[k].to_parquet(cache_paths[k])
        sel_day[k].to_frame("n").to_parquet(sd_paths[k])
        print(f"[cache] {k} saved ({len(panels[k]):,} rows)")
    return panels, sel_day


t1 = time.time()
panels_lt, sel_day_lt = _load_or_build_lt()
print(f"[panels LT] {time.time()-t1:.1f}s  keys={sorted(panels_lt)}")


split_data_at_dates: train_end=2025-06-01 val_end=None test_start=2025-06-01
  Train:    345,185 trades  (1,817 markets)
  Val:            0 trades  (    0 markets)
  Test:   6,043,259 trades  (19,026 markets)
  Total:  6,388,444 trades  (20,843 markets)
buys_lt=6,388,444 wallets_all=2,020
[cache] UNL loaded (66,135 rows)
[cache] UNL_B loaded (31,402 rows)
[cache] UNL_B_hhi loaded (31,331 rows)
[cache] UNL_p98 loaded (66,135 rows)
[cache] UNL_p95 loaded (66,135 rows)
[cache] UNL_p90 loaded (66,135 rows)
[panels LT] 0.1s  keys=['UNL', 'UNL_B', 'UNL_B_hhi', 'UNL_p90', 'UNL_p95', 'UNL_p98']


In [44]:
# Walk-forward scale + continuous $10k sim over the full horizon
def walk_forward_scales(selr, *, retune_days=14, lookback_days=120,
                        min_hist=200, default_scale=0.5, tune_max_rows=40_000):
    fr = selr.copy()
    rel = np.maximum(
        pd.to_datetime(fr["end_date_iso"], utc=True).values.astype("datetime64[ns]"),
        pd.to_datetime(fr["market_close"], utc=True).values.astype("datetime64[ns]"),
    )
    fr["rel"] = pd.DatetimeIndex(rel).tz_localize("UTC").floor("1D")
    fr = fr.sort_values("rel").reset_index(drop=True)
    rel_ns = fr["rel"].values.astype("datetime64[ns]").astype("int64")
    days = pd.Index(sorted(set(fr["day"].unique())))
    days = days[days >= LT_START]
    scale_map, tuned, tuned_at = {}, None, None
    for d in days:
        if tuned_at is None or (d - tuned_at).days >= retune_days:
            hist = fr.iloc[
                np.searchsorted(rel_ns, (d - pd.Timedelta(days=lookback_days)).value, side="left"):
                np.searchsorted(rel_ns, d.value, side="left")
            ]
            hist = hist[hist["copyable_qty"] > 0]
            if len(hist) >= min_hist:
                if len(hist) > tune_max_rows:
                    hist = hist.iloc[:tune_max_rows]
                best, _ = select_scale(hist, "score1", BUDGET, SCALE_GRID, 0.0,
                                       primary="sharpe_daily", alpha_col="alpha_equal")
                tuned, tuned_at = float(best), d
        scale_map[d] = tuned if tuned is not None else default_scale
    return scale_map


def lt_sim(fr):
    fr = fr.copy()
    fr = fr[fr["day"] >= LT_START]
    fr = fr[pd.to_datetime(fr["end_date_iso"], utc=True) <= DATA_END]
    scale_map = walk_forward_scales(fr)
    fr["alpha_wf"] = fr["day"].map(scale_map).fillna(0.5) * fr["alpha_equal"]
    res = capital_constrained_sim(fr[SIM_COLS + ["alpha_wf"]], "score1", BUDGET, 1.0,
                                  0.0, alpha_col="alpha_wf")
    daily = res["daily_pnl"].sort_index()
    cum = daily.cumsum()
    return dict(fr=fr, res=res, scale_map=scale_map, daily=daily, cum=cum,
                net_pnl=float(res["net_pnl"]), peak_used=float(res["peak_used"]),
                mean_used=float(res["mean_used"]),
                sharpe=round(sizing_sharpe(daily, 365.0), 3),
                trades=int(res["trades"]),
                max_dd=float((cum.cummax() - cum).max()) if len(cum) else 0.0)


LT_CONFIGS = [
    ("UNL_equal_p<0.1", "UNL", "p<0.1"),
    ("UNL_B_equal_p<0.1", "UNL_B", "p<0.1"),
    ("UNL_B_hhi_equal_p<0.1", "UNL_B_hhi", "p<0.1"),
    ("UNL_p98_equal_p<0.1", "UNL_p98", "p<0.1"),
    ("UNL_p95_equal_p<0.1", "UNL_p95", "p<0.1"),
    ("UNL_p90_equal_p<0.1", "UNL_p90", "p<0.1"),
    ("UNL_p98_equal_none", "UNL_p98", None),
    ("UNL_p95_equal_none", "UNL_p95", None),
    ("UNL_p90_equal_none", "UNL_p90", None),
    ("UNL_B_hhi_equal_none", "UNL_B_hhi", None),
]
t1 = time.time()
lt_out = {}
for name, key, overlay in LT_CONFIGS:
    fr = panels_lt[key].copy()
    if overlay == "p<0.1":
        fr = fr[fr["price"] < 0.1]
    lt_out[name] = lt_sim(fr)
    d = lt_out[name]
    print(f"[lt] {name}: pnl={d['net_pnl']:,.0f} sharpe={d['sharpe']} "
          f"trades={d['trades']:,} mean_used={d['mean_used']:,.0f} "
          f"max_dd={d['max_dd']:,.0f} ({time.time()-t1:.0f}s)")
    t1 = time.time()

lt_view = ["net_pnl", "sharpe", "trades", "peak_used", "mean_used", "max_dd"]
lt_summary = pd.DataFrame(
    {n: {k: (round(v, 2) if isinstance(v, float) else v) for k, v in d.items() if k in lt_view}
     for n, d in lt_out.items()}).T
print(lt_summary.round(2).to_string())

[lt] UNL_equal_p<0.1: pnl=1,800 sharpe=0.121 trades=4,590 mean_used=341 max_dd=5,352 (2s)
[lt] UNL_B_equal_p<0.1: pnl=31,393 sharpe=1.01 trades=2,289 mean_used=296 max_dd=4,907 (1s)
[lt] UNL_B_hhi_equal_p<0.1: pnl=31,394 sharpe=1.01 trades=2,287 mean_used=296 max_dd=4,905 (1s)
[lt] UNL_p98_equal_p<0.1: pnl=1,800 sharpe=0.121 trades=4,590 mean_used=341 max_dd=5,352 (2s)
[lt] UNL_p95_equal_p<0.1: pnl=1,800 sharpe=0.121 trades=4,590 mean_used=341 max_dd=5,352 (2s)
[lt] UNL_p90_equal_p<0.1: pnl=1,800 sharpe=0.121 trades=4,590 mean_used=341 max_dd=5,352 (2s)
[lt] UNL_p98_equal_none: pnl=19,580 sharpe=0.315 trades=22,361 mean_used=6,146 max_dd=11,741 (9s)
[lt] UNL_p95_equal_none: pnl=19,580 sharpe=0.315 trades=22,361 mean_used=6,146 max_dd=11,741 (9s)
[lt] UNL_p90_equal_none: pnl=19,580 sharpe=0.315 trades=22,361 mean_used=6,146 max_dd=11,741 (9s)
[lt] UNL_B_hhi_equal_none: pnl=28,313 sharpe=0.654 trades=11,948 mean_used=5,898 max_dd=8,172 (4s)
                        net_pnl  peak_used  mea

In [45]:
# Daily exposure + top-10 contributor wallets (primary config)
import plotly.graph_objects as go

prim = lt_out["UNL_B_hhi_equal_p<0.1"]
sub = prim["fr"].loc[prim["res"]["taken"]].copy()
qty = np.clip(1.0 * sub["score1"] * sub["alpha_wf"] * sub["copyable_qty"],
              0.0, sub["copyable_qty"])
cost = sub["price"] * qty
op = sub["dt"].dt.floor("4h")
cl = np.maximum(
    pd.to_datetime(sub["end_date_iso"], utc=True).values.astype("datetime64[ns]"),
    pd.to_datetime(sub["market_close"], utc=True).values.astype("datetime64[ns]"),
)
cl = pd.DatetimeIndex(cl).tz_localize("UTC").floor("4h")
day_idx = pd.date_range(prim["fr"]["dt"].min().floor("4h"),
                        prim["fr"]["dt"].max().floor("4h"), freq="4h", tz="UTC")
ev = pd.Series(0.0, index=day_idx)
ev = ev.add(pd.Series(cost.to_numpy(), index=op).groupby(level=0).sum(), fill_value=0.0)
ev = ev.add(-pd.Series(cost.to_numpy(), index=cl).groupby(level=0).sum(), fill_value=0.0)
expo = ev.cumsum()

fig = go.Figure()
for name, d in lt_out.items():
    fig.add_trace(go.Scatter(x=d["cum"].index, y=d["cum"].values, name=name, mode="lines"))
fig.update_layout(
    title=f"Long-term cumulative PnL — rolling copy, $10k, walk-forward scale ({LT_START.date()} → {DATA_END.date()})",
    template="plotly_dark", xaxis_title="resolution day", yaxis_title="cumulative PnL ($)",
)
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=expo.index, y=expo.values, name="capital locked", mode="lines"))
fig2.add_hline(y=BUDGET, line_dash="dot", line_color="red")
fig2.update_layout(
    title="Momentary exposure — UNL_B_hhi_equal_p<0.1 (peak = 100% of $10k)",
    template="plotly_dark", xaxis_title="day", yaxis_title="capital locked ($)",
)
fig2.show()

per_share = sub["copyable_pnl"] / sub["copyable_qty"].replace(0, np.nan)
pnl = (per_share * qty).fillna(0.0)
g = pd.DataFrame({"wallet": sub["wallet"].to_numpy(), "rel": cl, "pnl": pnl.to_numpy()})
pw = g.groupby(["wallet", "rel"])["pnl"].sum().unstack().fillna(0.0).cumsum(axis=1)
top10 = pw.T.iloc[-1].nlargest(10).index
pwtop = pw.T[top10]

fig3 = go.Figure()
for w in top10:
    fig3.add_trace(go.Scatter(x=pwtop.index, y=pwtop[w].values, name=str(w)[:12], mode="lines"))
fig3.update_layout(
    title="Top-10 contributing wallets — cumulative PnL (UNL_B_hhi_equal_p<0.1)",
    template="plotly_dark", xaxis_title="resolution day", yaxis_title="cumulative PnL ($)",
)
fig3.show()

print("top-10 wallets, final cumulative PnL:")
print(pwtop.iloc[-1].sort_values(ascending=False).round(0).to_string())
print(f"top-10 share of total: {pwtop.iloc[-1].sum():,.0f} / {pw.T.iloc[-1].sum():,.0f} "
      f"({pwtop.iloc[-1].sum() / pw.T.iloc[-1].sum():.0%})")

top-10 wallets, final cumulative PnL:
wallet
0x134a63b764ac7b008356e8db1857db94e6b09e42    30647.0
0xa0bca9bdd8540da95060ed1fafb78aa03835d428     4151.0
0xf658449199d0bcf544de0ff928a3b66685f3dcfe     1200.0
0x8f41129e43ebfbfe6075d0804f3b2bb763b3260e      948.0
0x41583f2efc720b8e2682750fffb67f2806fece9f      293.0
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62      167.0
0x7d51f58c68b427d113d5c84b508161d4e0286f1f      164.0
0x40cfb29411d29f4fa0908f2a121297042cccd21d      153.0
0x87d6a04aa4e48c6e7a7c9b009d70f1c79e7ed1e7      115.0
0x78a19803306d350cd1120bb07fcfe91a665be6bd      101.0
top-10 share of total: 37,939 / 31,394 (121%)


In [46]:
# Write the long-term record
lt_rows = []
for name, d in lt_out.items():
    sm = pd.Series(d["scale_map"])
    lt_rows.append({
        "config": name,
        "net_pnl": round(d["net_pnl"], 2),
        "sharpe": d["sharpe"],
        "trades": d["trades"],
        "peak_used": round(d["peak_used"], 2),
        "mean_used": round(d["mean_used"], 2),
        "max_dd": round(d["max_dd"], 2),
        "n_eval_days": int(len(sm)),
        "scale_p10": round(float(sm.quantile(0.1)), 2),
        "scale_median": round(float(sm.median()), 2),
        "scale_p90": round(float(sm.quantile(0.9)), 2),
    })
lt_df = pd.DataFrame(lt_rows)
lt_df.to_csv(HERE / "rc_politics_lt.csv", index=False)
(HERE / "rc_politics_lt_summary.json").write_text(json.dumps({
    "split": SPLIT_KWARGS_LT, "budget": BUDGET,
    "eval_start": str(LT_START.date()), "data_end": str(DATA_END.date()),
    "results": lt_df.to_dict(orient="records"),
    "top10_wallets": [str(w) for w in top10],
}, indent=2, default=str))
print(lt_df.round(2).to_string(index=False))
print("wrote rc_politics_lt.csv / rc_politics_lt_summary.json")

# ---- Top-market concentration + resolution-age lag (primary config) ----
prim = lt_out["UNL_B_hhi_equal_p<0.1"]
sub = prim["fr"].loc[prim["res"]["taken"]].copy()
qty = np.clip(sub["score1"] * sub["alpha_wf"] * sub["copyable_qty"],
              0.0, sub["copyable_qty"])
pnl = (sub["copyable_pnl"] / sub["copyable_qty"].replace(0, np.nan) * qty).fillna(0.0)
tm = pd.DataFrame({
    "condition_id": sub["condition_id"], "wallet": sub["wallet"],
    "copy_day": pd.to_datetime(sub["day"]),
    "end": pd.to_datetime(sub["end_date_iso"], utc=True),
    "pnl": pnl.to_numpy(),
})
long = tm[tm["pnl"] > 0]
pm = long.groupby("condition_id").agg(
    pnl=("pnl", "sum"), fills=("pnl", "size"), wallets=("wallet", "nunique"),
    copy_day=("copy_day", "max"), end=("end", "max"),
).sort_values("pnl", ascending=False)
pm["share_of_long"] = pm["pnl"] / long["pnl"].sum()
pm["cum_share"] = pm["share_of_long"].cumsum()
print("\n=== top 10 markets by copied PnL (long-only, +$%.0f total, %d markets) ===" % (
    long["pnl"].sum(), len(pm)))
print(pm.head(10).assign(
    pnl=lambda s: s["pnl"].map("${:,.0f}".format),
    share_of_long=lambda s: s["share_of_long"].map("{:.1%}".format),
    cum_share=lambda s: s["cum_share"].map("{:.1%}".format),
).to_string())

hold = tm.copy()
hold["hold_d"] = (hold["end"] - hold["copy_day"]).dt.days
buckets = [(0, 2), (3, 6), (7, 13), (14, 63), (64, None)]
lab = ["0-2d", "3-6d", "7-13d", "14-63d", "64d+"]
tot = tm["pnl"].sum()
rows = []
for (lo, hi), name in zip(buckets, lab):
    if hi is None:
        m = hold["hold_d"] >= lo
    else:
        m = (hold["hold_d"] >= lo) & (hold["hold_d"] <= hi)
    rows.append((name, hold.loc[m, "pnl"].sum(), int(m.sum())))
hd = pd.DataFrame(rows, columns=["days copy→resolution", "pnl", "fills"])
hd["share_of_net"] = hd["pnl"] / tot
print("\n=== PnL by days-from-copy to resolution (net %.0f) ===" % tot)
print(hd.assign(pnl=lambda s: s["pnl"].map("${:,.0f}".format),
                share_of_net=lambda s: s["share_of_net"].map("{:.1%}".format)).to_string())
wpnl = tm.groupby("wallet")["pnl"].sum()
print(f"\nnet-negative wallets: {int((wpnl < 0).sum())} of {wpnl.size}")


               config  net_pnl  sharpe  trades  peak_used  mean_used   max_dd  n_eval_days  scale_p10  scale_median  scale_p90
      UNL_equal_p<0.1  1800.48    0.12    4590    5437.07     341.50  5351.99          363        0.1           0.3        1.6
    UNL_B_equal_p<0.1 31392.82    1.01    2289    6104.08     295.74  4906.60          212        0.3           0.5        1.6
UNL_B_hhi_equal_p<0.1 31394.22    1.01    2287    6104.08     295.73  4905.21          211        0.3           0.5        1.6
  UNL_p98_equal_p<0.1  1800.48    0.12    4590    5437.07     341.50  5351.99          363        0.1           0.3        1.6
  UNL_p95_equal_p<0.1  1800.48    0.12    4590    5437.07     341.50  5351.99          363        0.1           0.3        1.6
  UNL_p90_equal_p<0.1  1800.48    0.12    4590    5437.07     341.50  5351.99          363        0.1           0.3        1.6
   UNL_p98_equal_none 19579.62    0.32   22361   10000.00    6146.49 11740.57          423        0.1          

## Long-term takeaways

*To be updated from the latest LT run.* Rolling USER point-in-time selection
over 2025-06 → 2026-08 with walk-forward scale; see the LT table and
`rc_politics_lt_summary.json` for the per-config record.